In [ ]:
try:
    from openmdao.utils.notebook_utils import notebook_mode  # noqa: F401
except ImportError:
    !python -m pip install openmdao[notebooks]

# pymooDriver

pymooDriver wraps the optimizer package pymoo, which provides a comprehensive suite of evolutionary and swarm-based single- and multi-objective optimization algorithms. Unlike gradient-based drivers, pymooDriver requires no derivatives from the model.

```{note}
The pymoo package does not come included with the OpenMDAO installation. It is a separate optional package that can be installed with:

    pip install pymoo
```

In this example, we use the DE (Differential Evolution) optimizer to minimize the objective of the Paraboloid problem.

In [ ]:
from openmdao.utils.notebook_utils import get_code
from myst_nb import glue
glue('code_pymoo_paraboloid', get_code('openmdao.test_suite.components.paraboloid.Paraboloid'), display=False)

:::{Admonition} `Paraboloid` class definition
:class: dropdown

{glue:}`code_pymoo_paraboloid`
:::

In [ ]:
import openmdao.api as om
from openmdao.test_suite.components.paraboloid import Paraboloid

prob = om.Problem()
model = prob.model

model.add_subsystem('comp', Paraboloid(), promotes=['*'])

prob.driver = om.pymooDriver()
prob.driver.options['optimizer'] = 'DE'
prob.driver.options['disp'] = False
prob.driver.run_settings['seed'] = 11
prob.driver.run_settings['termination'] = ('n_gen', 100)

model.add_design_var('x', lower=-50.0, upper=50.0)
model.add_design_var('y', lower=-50.0, upper=50.0)
model.add_objective('f_xy')

prob.setup()

prob.set_val('x', 50.0)
prob.set_val('y', 50.0)

prob.run_driver()

In [ ]:
print(prob.get_val('x'))
print(prob.get_val('y'))

In [ ]:
from openmdao.utils.assert_utils import assert_near_equal
assert_near_equal(prob.get_val('x'), 6.66666667, 1e-3)
assert_near_equal(prob.get_val('y'), -7.3333333, 1e-3)

The `optimizer` option selects which pymoo algorithm to use. pymooDriver supports
the following optimizers (case sensitive):

**Single-Objective:**
- `'GA'` - Genetic Algorithm
- `'DE'` - Differential Evolution
- `'BRKGA'` - Biased Random Key Genetic Algorithm
- `'NelderMead'` - Nelder-Mead simplex algorithm
- `'PatternSearch'` - Pattern search algorithm
- `'CMAES'` - Covariance Matrix Adaptation Evolution Strategy
- `'ES'` - Evolution Strategy
- `'SRES'` - Stochastic Ranking Evolution Strategy
- `'ISRES'` - Improved Stochastic Ranking Evolution Strategy
- `'NRBO'` - Newton-Raphson Based Optimizer
- `'MixedVariableGA'` - Genetic Algorithm with support for discrete (integer) and mixed integer design variables

**Multi-Objective:**
- `'NSGA2'` - Non-dominated Sorting Genetic Algorithm II
- `'NSGA3'` - Non-dominated Sorting Genetic Algorithm III
- `'UNSGA3'` - Unified NSGA-III
- `'MOEAD'` - Multi-Objective Evolutionary Algorithm Based on Decomposition
- `'AGEMOEA'` - Adaptive Geometry Estimation based MOEA
- `'CTAEA'` - Constrained Two-Archive Evolutionary Algorithm
- `'SMSEMOA'` - S-Metric Selection EMOA
- `'RVEA'` - Reference Vector Guided Evolutionary Algorithm
- `'CMOPSO'` - Constrained Multi-Objective Particle Swarm Optimization
- and others — see the [pymoo documentation](https://pymoo.org) for the full list.

## pymooDriver Options

In [ ]:
om.show_options_table('openmdao.drivers.pymoo_driver.pymooDriver')

## pymooDriver Constructor

The call signature for the *pymooDriver* constructor is:

```{eval-rst}
    .. automethod:: openmdao.drivers.pymoo_driver.pymooDriver.__init__
       :noindex:
```

## Using pymooDriver

pymooDriver has a small number of unified options that can be set as keyword arguments when it is instantiated or through the `options` dictionary. We have already shown how to set the `optimizer` option. The `disp` option controls whether pymooDriver prints a completion message when the optimization finishes and iteration level outputs.

In [ ]:
import openmdao.api as om
from openmdao.test_suite.components.paraboloid import Paraboloid

prob = om.Problem()
model = prob.model

model.add_subsystem('comp', Paraboloid(), promotes=['*'])

prob.driver = om.pymooDriver()
prob.driver.options['optimizer'] = 'DE'
prob.driver.options['disp'] = True
prob.driver.run_settings['seed'] = 11
prob.driver.run_settings['termination'] = ('n_gen', 400)

model.add_design_var('x', lower=-50.0, upper=50.0)
model.add_design_var('y', lower=-50.0, upper=50.0)
model.add_objective('f_xy')

prob.setup()
prob.run_driver()

## Algorithm Settings

Each pymoo algorithm has its own set of hyperparameters that control the structure of the algorithm — for example, population size, crossover and mutation operators. These are passed to the algorithm constructor via the `alg_settings` dictionary.

See the [pymoo documentation](https://pymoo.org) for the available settings for each algorithm.

Here, we set the population size for the GA using the `alg_settings`:

In [ ]:
import openmdao.api as om
from openmdao.test_suite.components.paraboloid import Paraboloid

prob = om.Problem()
model = prob.model

model.add_subsystem('comp', Paraboloid(), promotes=['*'])

prob.driver = om.pymooDriver()
prob.driver.options['optimizer'] = 'GA'
prob.driver.options['disp'] = False
prob.driver.alg_settings['pop_size'] = 50
prob.driver.run_settings['seed'] = 11
prob.driver.run_settings['termination'] = ('n_gen', 400)

model.add_design_var('x', lower=-50.0, upper=50.0)
model.add_design_var('y', lower=-50.0, upper=50.0)
model.add_objective('f_xy')

prob.setup()
prob.run_driver()

print(prob.get_val('x'))
print(prob.get_val('y'))

## Run Settings

Run-level settings that control how the optimization executes — such as the random seed, verbosity, termination criterion, and callback — are passed via the `run_settings` dictionary. These are forwarded to `pymoo.optimize.minimize()` and subsequently to `algorithm.setup()`.

Common run settings include:

- `'seed'` — integer random seed for reproducibility
- `'verbose'` — whether to print iteration-level progress (overrides `disp` option if set)
- `'termination'` — a pymoo termination object or shorthand tuple such as `('n_gen', 200)`
- `'save_history'` — whether to store per-generation history in the results object
- `'callback'` — a pymoo callback object called after each generation

Note that depending on which optimizer you choose, there may be slightly different run settings keys available, though it is not explicitly documented in pymoo which algorithms allow exactly which run settings. Here, we set a seed for reproducibility and a generation-based termination criterion in the `run_settings`:

In [ ]:
import openmdao.api as om
from openmdao.test_suite.components.paraboloid import Paraboloid

prob = om.Problem()
model = prob.model

model.add_subsystem('comp', Paraboloid(), promotes=['*'])

prob.driver = om.pymooDriver()
prob.driver.options['optimizer'] = 'DE'
prob.driver.options['disp'] = False
prob.driver.run_settings['seed'] = 42
prob.driver.run_settings['termination'] = ('n_gen', 400)
prob.driver.run_settings['verbose'] = False

model.add_design_var('x', lower=-50.0, upper=50.0)
model.add_design_var('y', lower=-50.0, upper=50.0)
model.add_objective('f_xy')

prob.setup()
prob.run_driver()

print(prob.get_val('x'))
print(prob.get_val('y'))

## Constrained Optimization

Constraints are supported by most pymoo algorithms (see the optimizer list above). Inequality and equality constraints defined via `add_constraint()` are automatically translated to pymoo's convention and passed to the algorithm.

A solution is considered successful if there is a feasible solution found in the final population. Note that pymoo hard codes a small tolerance for equality constraints (1e-4) into its constraint violation calculation by default.

Here, we minimize the Paraboloid subject to an inequality constraint `x + y >= 15`, which forces the optimum away from the unconstrained minimum:

In [ ]:
import openmdao.api as om
from openmdao.test_suite.components.paraboloid import Paraboloid

prob = om.Problem()
model = prob.model

model.add_subsystem('comp', Paraboloid(), promotes=['*'])
model.add_subsystem('con', om.ExecComp('g = x + y'), promotes=['*'])

prob.driver = om.pymooDriver()
prob.driver.options['optimizer'] = 'GA'
prob.driver.options['disp'] = False
prob.driver.run_settings['seed'] = 11
prob.driver.run_settings['termination'] = ('n_gen', 400)

model.add_design_var('x', lower=-50.0, upper=50.0)
model.add_design_var('y', lower=-50.0, upper=50.0)
model.add_objective('f_xy')
model.add_constraint('g', lower=15.0)

model.set_input_defaults('y', val=0.0)
model.set_input_defaults('x', val=0.0)

prob.setup()
prob.run_driver()

print(prob.get_val('x'))
print(prob.get_val('y'))
print('x + y =', prob.get_val('x') + prob.get_val('y'))

In [ ]:
from openmdao.utils.assert_utils import assert_near_equal

# Constraint x + y >= 15 should be satisfied
assert prob.get_val('x') + prob.get_val('y') >= 15.0 - 1e-3

## Integer and Mixed Integer Optimization

Discrete (integer) design variables are supported via the `MixedVariableGA` optimizer. This is the only pymoo algorithm that uses pymoo's mixed-variable-aware sampling and mating operators, which are required to correctly search integer and mixed integer/continuous spaces. Using any other optimizer with discrete design variables will raise an error.

In this example, we minimize a simple function of one continuous and one integer design variable. The optimal solution is at `x = 2.5`, `n = 3`:

In [ ]:
import openmdao.api as om


class MixedComp(om.ExplicitComponent):
    """Minimize f = (x - 2.5)^2 + (n - 3)^2 with x continuous and n integer."""

    def setup(self):
        self.add_input('x', val=0.0)
        self.add_discrete_input('n', val=0)
        self.add_output('f', val=0.0)

    def setup_partials(self):
        self.declare_partials('f', 'x')

    def compute(self, inputs, outputs, discrete_inputs, discrete_outputs):
        outputs['f'] = (inputs['x'] - 2.5)**2 + (discrete_inputs['n'] - 3)**2

    def compute_partials(self, inputs, partials, discrete_inputs):
        partials['f', 'x'] = 2.0 * (inputs['x'] - 2.5)


prob = om.Problem()
prob.model.add_subsystem('comp', MixedComp())

prob.driver = om.pymooDriver()
prob.driver.options['optimizer'] = 'MixedVariableGA'
prob.driver.options['disp'] = False
prob.driver.run_settings['seed'] = 11
prob.driver.run_settings['termination'] = ('n_gen', 200)

prob.model.add_design_var('comp.x', lower=-10.0, upper=10.0)
prob.model.add_design_var('comp.n', lower=0, upper=10)
prob.model.add_objective('comp.f')

prob.setup()
prob.run_driver()

print('x =', prob.get_val('comp.x'))
print('n =', prob.get_val('comp.n'))

In [ ]:
from openmdao.utils.assert_utils import assert_near_equal

assert_near_equal(prob.get_val('comp.x'), 2.5, 1e-2)
assert prob.get_val('comp.n') == 3

## Multi-Objective Optimization

pymooDriver supports multi-objective optimization through pymoo's Pareto-based algorithms such as NSGA2. When a multi-objective optimizer is used, the model is left at the state of the last function evaluation after `run_driver()` completes. The full Pareto front is stored on the driver in `driver.pareto`, which contains:

- `driver.pareto['X']` — design variable values for each Pareto-optimal solution, shape `(n_solutions, n_vars)`
- `driver.pareto['F']` — objective values for each Pareto-optimal solution, shape `(n_solutions, n_objs)`

In this example, we minimize two competing objectives over a single design variable:

In [ ]:
import numpy as np
import openmdao.api as om

prob = om.Problem()
model = prob.model

# Two competing objectives: f1 = x^2, f2 = (x - 2)^2
# Pareto front spans x in [0, 2]
exec_comp = model.add_subsystem('exec', om.ExecComp(['f1 = x**2', 'f2 = (x - 2.0)**2'],
                                                    x=0.0, f1=0.0, f2=0.0))

prob.driver = om.pymooDriver()
prob.driver.options['optimizer'] = 'NSGA2'
prob.driver.options['disp'] = False
prob.driver.run_settings['seed'] = 42
prob.driver.run_settings['termination'] = ('n_gen', 200)

model.add_design_var('exec.x', lower=0.0, upper=3.0)
model.add_objective('exec.f1')
model.add_objective('exec.f2')

prob.setup()
prob.run_driver()

In [ ]:
from pymoo.visualization.scatter import Scatter

X = prob.driver.pareto['X']
F = prob.driver.pareto['F']

plot = Scatter()
plot.add(prob.driver.pareto['F'], facecolor="none", edgecolor="red")
plot.show()

In [ ]:
from openmdao.utils.assert_utils import assert_near_equal

# Pareto front should span x in [0, 2]
assert X is not None
assert F is not None
assert_near_equal(X.min(), 0.0, 1e-2)
assert_near_equal(X.max(), 2.0, 1e-2)

The raw pymoo result object is also available at `prob.driver.pymoo_results` for any algorithm so users can access additional information such as execution time, algorithm state, and population history (if `save_history=True` was set in `run_settings`).